<div dir="rtl">

# לבקש נתונים משירות: API ו־JSON

</div>

<div dir="rtl">

בשיעור הזה נלמד מהו API, איך תוכנית מבקשת ממנו נתונים עם פרמטרים וסוכן משתמש, איך היא בודקת את קוד המצב, ואיך היא קוראת תשובה ב־JSON: מילונים ורשימות מקוננים, ומפתח שחסר.

## תקציר ומספר קישורים למונח חומרה

נניח שאנחנו רוצים מילון לימוד של מונחי חומרה: לכל מונח תקציר קצר אחד ומספר הדפים האחרים שהדף שלו מקשר אליהם. דף של ויקיפדיה כתוב בשביל קורא: לצד הטקסט יש בו טבלת עובדות, מספרי הפניות בסוגריים מרובעים ותפריטים, ו[Beautiful Soup](https://dvirross.com/he/courses/intro-data-science/data-collection-beautifulsoup) הייתה חותכת כל אחד מהם, חלק אחרי חלק. אתר ויקיפדיה מציע דרך אחרת: [API](https://dvirross.com/he/courses/intro-data-science/glossary#api) (application programming interface, ממשק תכנות), דרך ששירות מציע לתוכנות כדי לבקש נתונים. דף נכתב בשביל אדם, עם תגיות ועיצוב. API עונה לתוכנית עם הנתונים עצמם, בפורמט שתוכנית קוראת בלי לחתוך שום דבר.

הוויקיפדיה האנגלית עונה לשאלות כאלה בכתובת אחת, `https://en.wikipedia.org/w/api.php`, והכללים שלה ב[תיעוד של ה־API](https://www.mediawiki.org/wiki/API:Main_page). כל שירות כותב כללים משלו: על אילו שאלות הוא עונה, איך כותבים שאלה, ואיך נראית התשובה.

</div>

<div dir="rtl">

## הכנה: הספריות ופונקציית הבדיקה

אין צורך בחשבון ואין צורך במפתח. התא הבא מייבא את `requests` (ב־Colab היא קיימת; התא מתקין אותה אם בסביבה אין). התשובות שהקוד קורא הן עותקים שמורים של תשובות אמיתיות של ה־API של ויקיפדיה האנגלית, ששמורים באתר הציבורי של הקורס, ולכן המספרים זהים לכולם. התא שמסומן **השירות החי** קורא לוויקיפדיה עצמה, ולכן המספרים בו עשויים להיות שונים מהמספרים בדף, והוא עלול להיכשל כשהשירות לא עונה או מבקש להאט. הוא כבוי (`RUN_LIVE = False`), כדי שכיתה שלמה לא תשלח בבת אחת בקשות לוויקיפדיה מאותה כתובת.

</div>

In [1]:
try:
    import requests
except ImportError:
    !pip install -q requests
    import requests
print('requests', requests.__version__)

requests 2.34.2


<div dir="rtl">

התא הבא מגדיר את הפונקציה `check(task_number)`. אחרי שמריצים את התא של משימה, מריצים את התא שמתחתיו: `check` מריצה שוב את התא שלכם, אוספת את מה שהוא מדפיס ומשווה אותו לפלט הצפוי. היא אומרת אם הפלט נכון, ואם לא באיזו שורה הוא שונה; היא לא מראה את הפתרון.

</div>

In [2]:
import ast, contextlib, hashlib, io

# The expected output of each task, as hashes of its lines (the solutions are not stored here).
EXPECTED = {int(k): v for k, v in {
    '1': [
        '6098f90d6d7f'
    ],
    '2': [
        'e8df814b551f'
    ],
    '3': [
        'b17ef6d19c7a'
    ],
    '4': [
        'e16aa6ff26b8',
        '2ea60c2384a6'
    ]
}.items()}


def check(task):
    """Run the last cell that ran (not a check) again and compare what it prints with the expected output of the task."""
    if task not in EXPECTED:
        print('אין משימה כזו: ', task)
        return
    cells = [s for s in In[:-1] if s.strip() and not s.lstrip().startswith('check(')]   # In: the cells that ran, kept by IPython
    tree = ast.parse(cells[-1] if cells else '')
    last = tree.body.pop() if tree.body and isinstance(tree.body[-1], ast.Expr) else None   # a bare last expression counts as printed
    out = io.StringIO()
    try:
        with contextlib.redirect_stdout(out):
            exec(compile(tree, '<task>', 'exec'), globals())
            if last is not None:
                value = eval(compile(ast.Expression(last.value), '<task>', 'eval'), globals())
                if value is not None:
                    print(repr(value))
    except Exception as error:
        print('התא נכשל בשגיאה:', type(error).__name__ + ':', error)
        return
    got = [hashlib.sha256(line.rstrip().encode()).hexdigest()[:12] for line in out.getvalue().rstrip().splitlines()]
    want = EXPECTED[task]
    if not got:
        print('לא הודפס כלום. הדפיסו את התשובה בתא המשימה, הריצו אותו, והריצו את `check` מיד אחריו.')
    elif len(got) != len(want):
        print('התא הדפיס {got} שורות והמשימה מצפה ל־{want}. קראו שוב את המשימה ואת הרמז.'.format(got=len(got), want=len(want)))
    elif got != want:
        print('שורה {k} שונה מהפלט הצפוי. קראו שוב את המשימה ואת הרמז.'.format(k=next(i for i in range(len(want)) if got[i] != want[i]) + 1))
    else:
        print('OK: הפלט כצפוי.')


<div dir="rtl">

## תשובות שמורות של השירות האמיתי

הדפדפן מתיר לדף באתר הזה לפנות רק לאתר הזה, ולכן התיבות של הדף לא יכולות לקרוא לוויקיפדיה. את התשובות ביקשנו מהשירות האמיתי פעם אחת, ב־16 בנובמבר 2026, ושמרנו אותן בלי שינוי כשישה קבצים באתר הזה. כל קובץ מחזיק את התשובה לשאלה אחת, ולכן הפרמטרים שנכתוב בתיבות לא משנים אותה. בשירות האמיתי הקריאה זהה, עם הכתובת `https://en.wikipedia.org/w/api.php` ועם אותם פרמטרים. המחברות מריצות את אותו קוד על העותקים השמורים; התא שלהן לשירות האמיתי כבוי (`RUN_LIVE = False`), כי כיתה חולקת כתובת אחת והשירות מבקש מתוכנות לשלוח מעט בקשות.

הטקסט של ויקיפדיה זמין ברישיון [Creative Commons Attribution-ShareAlike 4.0](https://creativecommons.org/licenses/by-sa/4.0/) (CC BY-SA 4.0). הכותרות, התקצירים, רשימות הקישורים וגדלי הדפים בתשובות השמורות הם מהוויקיפדיה האנגלית, [en.wikipedia.org](https://en.wikipedia.org), ונשלפו ב־16 בנובמבר 2026 דרך ה־API שלה; המחברים שלהם הם העורכים של ויקיפדיה, שההיסטוריה של כל דף מפרטת אותם. הקבצים לא שונו, וטקסט שלוקחים מהם חייב לשמור את הייחוס הזה ואת אותו רישיון.

## נקודת קצה ופרמטרים

הכתובת שאליה תוכנית שולחת את השאלות שלה היא [**נקודת קצה**](https://dvirross.com/he/courses/intro-data-science/glossary#endpoint) (endpoint). שאלה נכתבת כ[**פרמטרים של שאילתה**](https://dvirross.com/he/courses/intro-data-science/glossary#query-parameter) (query parameters): זוגות של שם וערך שמוסיפים לכתובת אחרי `?` ומפרידים ביניהם ב־`&`. הפונקציה `requests.get` מקבלת את הפרמטרים כמילון בארגומנט `params`, ובונה מהם את הכתובת הסופית. בשאלה הראשונה הפרמטרים מבקשים את ההקדמה של הדף על מערך שערים לוגיים שמתכנתים בשטח (FPGA). התכונה `url` של התשובה היא הכתובת שנשלחה, ו־`split('?')[1]` שומרת את הטקסט שאחרי סימן השאלה.

</div>

In [3]:
import requests

copies = (
    'https://dvirross.com/materials/courses/'
    'intro-data-science/'
)
summary_params = {
    'action': 'query', 'format': 'json', 'formatversion': 2,
    'prop': 'extracts', 'exintro': 1, 'explaintext': 1,
    'titles': 'Field-programmable gate array',
}
response = requests.get(
    copies + 'wikipedia-fpga-summary.json',
    params=summary_params,
)
print(response.url.split('?')[1])

action=query&format=json&formatversion=2&prop=extracts&exintro=1&explaintext=1&titles=Field-programmable+gate+array


<div dir="rtl">

הספרייה כתבה כל זוג כ־`name=value` והחליפה את הרווחים של הכותרת ב־`+`, כי בכתובת אין רווחים.

## מה הפרמטרים אומרים

| פרמטר | ערך | משמעות |
| --- | --- | --- |
| `action` | `query` | לבקש נתונים על דפים |
| `format` | `json` | לכתוב את התשובה ב־JSON |
| `formatversion` | `2` | המבנה החדש והפשוט יותר של התשובה |
| `prop` | `extracts` | התכונה של הדף שאנחנו רוצים: תקציר טקסט |
| `exintro` | `1` | רק ההקדמה, הטקסט שלפני הכותרת הראשונה |
| `explaintext` | `1` | טקסט פשוט, בלי תגיות |
| `titles` | `Field-programmable gate array` | הכותרת של הדף |

בשירות האמיתי אותה שאלה היא קריאה לנקודת הקצה של ויקיפדיה. היא מוצגת כאן ולא מורצת, כי התיבות של הדף לא מגיעות אליה:

```python
response = requests.get(
    'https://en.wikipedia.org/w/api.php',
    params=summary_params,
)
```

הכתובת הסופית ש־`requests` שולחת היא נקודת הקצה, `?` והפרמטרים, הכול בשורה אחת. אותה כתובת בשורת הכתובת של דפדפן מציגה את התשובה כטקסט.

```text
https://en.wikipedia.org/w/api.php?action=query&format=json&formatversion=2&prop=extracts&exintro=1&explaintext=1&titles=Field-programmable+gate+array
```

## קוד המצב

כל תשובה של שרת מתחילה ב[**קוד מצב**](https://dvirross.com/he/courses/intro-data-science/glossary#status-code) (status code), שתוכנית בודקת לפני שהיא קוראת את השאר. התכונה `status_code` של התשובה מחזיקה אותו. בקשה לקובץ שלא קיים באתר הזה מחזירה תשובה עם הקוד 404.

</div>

In [4]:
print(response.status_code)

missing = requests.get(copies + 'wikipedia-no-such-copy.json')
print(missing.status_code)

200
404


<div dir="rtl">

אלה הקודים שתוכנית פוגשת ב־API:

| קוד | משמעות ומה התוכנית עושה |
| --- | --- |
| 200 | התשובה הגיעה: קוראים אותה |
| 403 | השירות מסרב: בודקים את סוכן המשתמש ואת הכללים |
| 404 | אין כתובת כזו: בודקים את נקודת הקצה |
| 429 | יותר מדי בקשות: ממתינים; הכותרת `Retry-After` אומרת כמה זמן |

תוכנית שלא בודקת את הקוד קוראת הודעת שגיאה כאילו היא נתונים. היא בודקת עם `if response.status_code != 200:` (`!=` פירושו "לא שווה") ועוצרת עם הודעה.

## כותרות וסוכן משתמש

בקשה נושאת, חוץ מהפרמטרים, גם [**כותרות**](https://dvirross.com/he/courses/intro-data-science/glossary#header) (headers): פיסות מידע על הבקשה עצמה, שנמסרות ל־`requests.get` כמילון בארגומנט `headers`. הכותרת `User-Agent` היא השם של התוכנית ששולחת את הבקשה, [**סוכן המשתמש**](https://dvirross.com/he/courses/intro-data-science/glossary#user-agent) (user agent). הספרייה שולחת את השם `python-requests` אלא אם כותבים שם אחר, וקרן ויקימדיה, שמפעילה את ויקיפדיה, מבקשת ב[מדיניות סוכן המשתמש](https://foundation.wikimedia.org/wiki/Policy:Wikimedia_Foundation_User-Agent_Policy) שלה שם שאומר מהי התוכנית ואיך אפשר להגיע לבעלים שלה. הארגומנט `timeout` הוא מספר השניות להמתין לתשובה לפני שמוותרים; בלעדיו תוכנית יכולה להמתין לנצח.

</div>

In [5]:
headers = {
    'User-Agent': (
        'IntroDataScienceCourse/1.0 '
        '(teaching example; contact via dvirross.com)'
    )
}
response = requests.get(
    copies + 'wikipedia-fpga-summary.json',
    params=summary_params,
    headers=headers,
    timeout=30,
)
if response.status_code != 200:
    print('The service answered', response.status_code)
else:
    print('The answer arrived')

The answer arrived


<div dir="rtl">

העותק השמור לא קורא את הכותרת; השירות האמיתי קורא. תוכנית מנומסת גם שואלת מעט שאלות, אחת בכל פעם, ושומרת את התשובות, כמו שהעותקים השמורים של השיעור עושים, במקום לשאול שוב.

## התשובה היא JSON

התשובה של ה־API היא טקסט בפורמט שנקרא [JSON](https://dvirross.com/he/courses/intro-data-science/glossary#json) (JavaScript Object Notation). הוא כתוב עם סוגריים מסולסלים שמחזיקים שמות וערכים, וסוגריים מרובעים שמחזיקים רשימות, אותם מילונים ורשימות כמו ב־Python. הטקסט של התשובה הוא `response.text`, מחרוזת.

השיטה `json()` של התשובה קוראת את הטקסט ומחזירה ערכים של Python: מילון בשביל סוגריים מסולסלים, רשימה בשביל סוגריים מרובעים, וטקסט, מספרים, `True` ו־`None` בשביל השאר. הפונקציה `type` אומרת מהו הסוג של ערך, ולמילון יש שיטה `keys()` שמחזירה את המפתחות שלו, השמות, ש־`list` הופכת לרשימה.

</div>

In [6]:
data = response.json()
print(type(response.text), type(data))
print(list(data.keys()))

<class 'str'> <class 'dict'>
['batchcomplete', 'query']


<div dir="rtl">

## מילונים ורשימות, אחד בתוך השני

בתשובה יש מילון בתוך מילון בתוך רשימה. הנה כל התשובה, עם הטקסט הארוך של התקציר מקוצר:

```json
{
  "batchcomplete": true,
  "query": {
    "pages": [
      {
        "pageid": 10969,
        "ns": 0,
        "title": "Field-programmable gate array",
        "extract": "A field-programmable gate array (FPGA) is a type of ..."
      }
    ]
  }
}
```

מגיעים לערך שבאמצע צעד אחרי צעד, עם סוגריים מרובעים: מפתח למילון, וברשימה ה**ציין** (index), המקום של הפריט, שנספר מ־0. `data['query']` הוא מילון; `['pages']` בתוכו הוא רשימה עם פריט אחד לכל כותרת ששאלנו עליה; `[0]` הוא הפריט הראשון שלה, מילון של דף אחד; ו־`['title']` ו־`['extract']` הם הערכים שלו. השיטה `split('. ')` חותכת טקסט בכל נקודה שאחריה רווח, ולכן `[0]` של התוצאה שלה הוא המשפט הראשון.

</div>

In [7]:
pages = data['query']['pages']
print(type(pages), len(pages))

page = pages[0]
print(page['title'])
print(len(page['extract']))
print(page['extract'].split('. ')[0])

<class 'list'> 1
Field-programmable gate array
1951
A field-programmable gate array (FPGA) is a type of configurable integrated circuit that can be repeatedly programmed after manufacturing


<div dir="rtl">

התקציר של הדף הוא 1,951 תווים, והמשפט הראשון שלו אומר מהו FPGA. התוכנית לא חתכה שום דבר מתוך תגיות: היא ירדה במבנה לפי שמות ומקומות.

### תרגול

**נסו בעצמכם.** המשתנה `data` מחזיק את התשובה על מערך השערים הלוגיים שמתכנתים בשטח. הדפיסו את אורך התקציר של הדף בתווים ואחריו, אחרי רווח, את מספר המילים שלו: מספר המילים ש־`split()` מוצאת ב־`extract`.

</div>

In [8]:
# Your turn

<div dir="rtl">

**פלט צפוי (המספרים מוסתרים)**

```text
<number> <number>
```

</div>

In [9]:
check(1)

<div dir="rtl">

> **רמז**
>
> רדו במבנה התשובה כמו בתיבה שלמעלה, עד ה־`extract` של הדף הראשון. השיטה `split()` בלי ארגומנט חותכת טקסט ברווחים ובמעברי שורה ומחזירה רשימה של המילים שלו, ו־`len` סופרת רשימה.

</div>

<div dir="rtl">

> **ללימוד עצמי, חשיבות 1.** קטע ללימוד עצמי: הוא אינו נלמד בהרצאה, ונקרא אותו בעצמנו. שאלה אחת על כמה דפים, ומציאת הגדול והקטן ביותר לפי מפתח. הקטע מסייע בנושא: שאלה אחת על כמה פריטים והשוואה ביניהם.

</div>

<div dir="rtl">

## כמה דפים בשאלה אחת

הפרמטר `titles` מקבל כמה כותרות, מופרדות ב־`|`. התכונה `info` מחזירה עובדות על דף, כמו `length`, הגודל של הטקסט שלו בתווים. בתשובה יש רשימה עם מילון אחד לכל דף, ולולאת `for` עוברת עליה.

</div>

In [10]:
info_params = {
    'action': 'query', 'format': 'json', 'formatversion': 2,
    'prop': 'info',
    'titles': (
        'RISC-V|Hardware description language|'
        'Field-programmable gate array'
    ),
}
info = requests.get(
    copies + 'wikipedia-pages-info.json',
    params=info_params,
).json()
info_pages = info['query']['pages']
for page in info_pages:
    print(page['title'], page['length'])

Field-programmable gate array 68061
Hardware description language 36937
RISC-V 170750


<div dir="rtl">

הדפים לא בסדר של השאלה: השירות מחזיר אותם בסדר שלו, כאן אלפביתי. לכן תוכנית מוצאת דף לפי הכותרת שלו ולא לפי המקום שלו ברשימה.

**נסו בעצמכם.** הרשימה `info_pages` מחזיקה מילון לכל אחד משלושת הדפים. מצאו את הדף הארוך ביותר ואת הדף הקצר ביותר לפי `length`, והדפיסו את הכותרת של הדף הארוך ביותר ואחריה, אחרי רווח, פי כמה הוא ארוך מהדף הקצר ביותר, מעוגל לספרה אחת אחרי הנקודה העשרונית.

</div>

In [11]:
# Your turn

<div dir="rtl">

**פלט צפוי (המספרים מוסתרים)**

```text
RISC-V <number>.<number>
```

</div>

In [12]:
check(2)

<div dir="rtl">

> **רמז**
>
> הפונקציה `max` מחזירה את הפריט עם הערך הגדול ביותר, והארגומנט `key` שלה אומר במה להשוות: `max(info_pages, key=lambda page: page['length'])`, כש־`lambda page: page['length']` היא פונקציה קטנה שמקבלת דף ומחזירה את האורך שלו. הפונקציה `min` פועלת באותו אופן.

</div>

<div dir="rtl">

## הקישורים של דף

התכונה `links` מחזירה את הדפים שדף מקשר אליהם. הפרמטר `pllimit` הוא המספר הגדול ביותר של קישורים בתשובה אחת, ו־`max` מבקש את הגדול ביותר שהשירות מתיר; `plnamespace` שערכו `0` משאיר רק קישורים למאמרים, ולא לדפי עזרה או דיון. בתשובה יש, בתוך הדף, רשימה `links`, וכל קישור הוא מילון עם `title`.

</div>

In [13]:
links_params = {
    'action': 'query', 'format': 'json', 'formatversion': 2,
    'prop': 'links', 'pllimit': 'max', 'plnamespace': 0,
    'titles': 'Field-programmable gate array',
}
links_data = requests.get(
    copies + 'wikipedia-fpga-links.json',
    params=links_params,
).json()
links = links_data['query']['pages'][0]['links']
print(len(links))
print(links[0])

500
{'ns': 0, 'title': '20-sim'}


<div dir="rtl">

בתשובה יש 500 קישורים, ו־500 הוא הגבול של תשובה אחת לתוכנית בלי הרשאות מיוחדות, לא מספר הקישורים של הדף. התשובה אומרת זאת בעצמה: יש בה מפתח `continue`, שמחזיק את המקום שבו התשובה הבאה תתחיל. האופרטור `in` בודק אם מפתח נמצא במילון.

</div>

In [14]:
print('continue' in links_data)
print(links_data['continue']['plcontinue'])

True
10969|0|Power_engineering_software


<div dir="rtl">

הקישורים מסודרים לפי האלפבית, והתשובה הזו נעצרת ב־"Power engineering software". לכן המספר 500 מתאר את החלק הראשון של הקישורים, מתחילת האלפבית עד האות P, ותוכנית שמדווחת "בדף יש 500 קישורים" טועה. כדי לקבל את השאר חוזרים על הבקשה עם הערך של `continue` שמתווסף לפרמטרים שלה.

השיטה `lower()` מחזירה טקסט באותיות קטנות, ו־`in` בודק גם אם טקסט מכיל טקסט אחר: `'logic' in 'Logic gate'` הוא `False`, כי האות הגדולה היא תו אחר, ו־`'logic' in 'Logic gate'.lower()` הוא `True`.

**נסו בעצמכם.** בין הקישורים ב־`links`, כמה מהם יש להם כותרת שמכילה את המילה `logic`, באות גדולה או בלעדיה? הדפיסו כמה יש, כמספר אחד.

</div>

In [15]:
# Your turn

<div dir="rtl">

**פלט צפוי (המספרים מוסתרים)**

```text
<number>
```

</div>

In [16]:
check(3)

<div dir="rtl">

> **רמז**
>
> בנו רשימה עם הבנת רשימה, `[link['title'] for link in links if ...]`, שהתנאי שלה בודק את הכותרת באותיות קטנות: `'logic' in link['title'].lower()`. הדפיסו את האורך של הרשימה.

</div>

<div dir="rtl">

## מפתח שחסר

לא כל דף ששואלים עליו קיים. השאלה הבאה מבקשת את התקצירים של שתי כותרות: `RISC-V`, ו־`RISC-VI`, כותרת שאין לה דף. עבור הכותרת החסרה השירות עונה בפריט שיש בו המפתח `missing` ואין בו המפתח `extract`, והוא ראשון ברשימה.

</div>

In [17]:
missing_params = {
    'action': 'query', 'format': 'json', 'formatversion': 2,
    'prop': 'extracts', 'exintro': 1, 'explaintext': 1,
    'exlimit': 'max', 'titles': 'RISC-V|RISC-VI',
}
answer = requests.get(
    copies + 'wikipedia-summary-missing.json',
    params=missing_params,
).json()
missing_pages = answer['query']['pages']
print(len(missing_pages))
print(missing_pages[0])

2
{'ns': 0, 'title': 'RISC-VI', 'missing': True}


<div dir="rtl">

קריאה של מפתח שלא קיים עם סוגריים מרובעים עוצרת את התוכנית עם `KeyError`. זה מה ש־`missing_pages[0]['extract']` עושה, ותוכנית שקוראת הרבה דפים נעצרת בדף החסר הראשון:

```text
KeyError: 'extract'
```

השיטה `get` של מילון מקבלת מפתח וערך ברירת מחדל, ומחזירה את ברירת המחדל במקום שגיאה כשהמפתח חסר. עם ארגומנט אחד, `get('extract')`, ברירת המחדל היא `None`. ברירת מחדל מהסוג הנכון מאפשרת לשאר התוכנית להמשיך: טקסט ריק `''` בשביל טקסט.

</div>

In [18]:
for page in missing_pages:
    text = page.get('extract', 'no such page')
    print(page['title'], '|', text.split('. ')[0])

RISC-VI | no such page
RISC-V | RISC-V (pronounced "risk-five") is a free and open standard instruction set architecture (ISA) for computer CPUs


<div dir="rtl">

אותה זהירות נדרשת גם לתשובה שלמה. שאלה שהשירות לא יכול לענות עליה, כאן `action` שלא קיים, מקבלת גם היא את קוד המצב 200, ובתשובה יש מפתח `error` במקום המפתח `query`:

</div>

In [19]:
bad = requests.get(
    copies + 'wikipedia-error.json',
    params={'action': 'nonsense', 'format': 'json'},
).json()
print(bad.get('query', 'no query in the answer'))
print(bad['error']['info'])

no query in the answer
Unrecognized value for parameter "action": nonsense.


<div dir="rtl">

תוכנית שקוראת API בודקת קודם את קוד המצב, ואחר כך בודקת שיש בתשובה המפתח שהיא צריכה.

**נסו בעצמכם.** לכל דף ב־`missing_pages`, הדפיסו את הכותרת שלו ואחריה, אחרי רווח, את מספר המילים של התקציר שלו: מספר המילים ש־`split()` מוצאת ב־`extract`. לדף שאין לו תקציר יש 0 מילים.

</div>

In [20]:
# Your turn

<div dir="rtl">

**פלט צפוי (המספרים מוסתרים)**

```text
RISC-VI 0
RISC-V <number>
```

</div>

In [21]:
check(4)

<div dir="rtl">

> **רמז**
>
> קחו את ה־`extract` עם `get` וברירת המחדל `''`, טקסט ריק, וחתכו אותו: `len(page.get('extract', '').split())`. השיטה `split()` של טקסט ריק מחזירה רשימה ריקה, שהאורך שלה הוא 0.

</div>

<div dir="rtl">

> **ללימוד עצמי, חשיבות 2.** קטע ללימוד עצמי: הוא אינו נלמד בהרצאה, ונקרא אותו בעצמנו. שאלת חיפוש, חזרה על בקשה כדי לקבל את כל הקישורים של דף, וחבילה של Python ל־API של ויקיפדיה. הקטע מסייע בנושא: קריאת תוצאות חיפוש, איסוף החלקים של תשובה שמגיעה בחלקים, ושימוש בחבילה שפונה ל־API.

</div>

<div dir="rtl">

## שאלת חיפוש

התשובה לחיפוש היא גם רשימה. הפרמטר `list` שערכו `search` מבקש מהשירות לחפש את המילים של `srsearch`, ו־`srlimit` הוא מספר התוצאות בתשובה. בתשובה יש, חוץ מהתוצאות, גם מספר כל ההתאמות, `totalhits`, בתוך המילון `searchinfo`.

</div>

In [22]:
search_params = {
    'action': 'query', 'format': 'json', 'formatversion': 2,
    'list': 'search', 'srlimit': 5,
    'srsearch': 'hardware description language',
}
search = requests.get(
    copies + 'wikipedia-search-hdl.json',
    params=search_params,
).json()
print(search['query']['searchinfo']['totalhits'])
for hit in search['query']['search']:
    print(hit['title'], hit['wordcount'])

18047
Hardware description language 3718
VHDL 4278
Altera Hardware Description Language 275
Ruby (hardware description language) 267
Verilog 4227


<div dir="rtl">

בתשובה יש את חמשת הדפים הראשונים מתוך 18,047 שמתאימים למילים. לכל התאמה יש גם `snippet`, פיסה מהטקסט שהמילים שנמצאו בה עטופות בתגיות HTML.

## כל הקישורים של דף

כדי לקבל את כל הקישורים של דף, התוכנית חוזרת על הבקשה. תשובה שיש בה מפתח `continue` אומרת איפה הבאה מתחילה: הערכים שלו מתווספים לפרמטרים של הבקשה הבאה, והלולאה נעצרת כשבתשובה אין אותו. הקוד מוצג ולא מורץ, כי העותק השמור מחזיק רק את התשובה הראשונה. הקריאה `{**links_params, **extra}` כותבת את הזוגות של שני מילונים אל מילון חדש אחד, ו־`time.sleep(1)` עוצרת לשנייה בין הבקשות, כדי להישאר מנומסים.

```python
import time

all_links = []
extra = {}
while True:
    answer = requests.get(
        'https://en.wikipedia.org/w/api.php',
        params={**links_params, **extra},
        headers=headers,
        timeout=30,
    ).json()
    page = answer['query']['pages'][0]
    all_links += [link['title'] for link in page.get('links', [])]
    if 'continue' not in answer:
        break
    extra = answer['continue']
    time.sleep(1)
print(len(all_links))
```

מספר הקישורים של דף משתנה בכל עריכה שלו, ולכן תוכנית שמריצה את הקוד מקבלת את המספר של אותו יום.

## חבילה שמבקשת במקומנו

לחלק מהשירותים יש חבילה של Python שכותבת את הבקשות במקומנו. לוויקיפדיה יש `Wikipedia-API`, שמייבאים בשם `wikipediaapi`. הקוד מוצג ולא מורץ, כי הוא פונה לשירות. החבילה בונה את הכתובת, שולחת את סוכן המשתמש שאנחנו נותנים לה, ומחזירה את הדף כאובייקט עם תקציר וקישורים.

```python
import wikipediaapi

wiki = wikipediaapi.Wikipedia(
    user_agent=(
        'IntroDataScienceCourse/1.0 '
        '(teaching example; contact via dvirross.com)'
    ),
    language='en',
)
page = wiki.page('Field-programmable gate array')
print(page.summary[:200])
print(len(page.links))
```

חבילה קצרה יותר, והיא מסתירה את הפרמטרים ואת המבנה של התשובה; שירות שאין לו חבילה עדיין בהישג יד של תוכנית שמבצעת את הקריאה בעצמה.

## נקודות לזכור

- API הוא דרך ששירות מציע לתוכנות כדי לבקש נתונים; התשובה שלו היא נתונים, לא דף שחותכים מתוך תגיות.
- שאלה היא נקודת קצה ופרמטרים של שאילתה: `requests.get(url, params={...})` כותבת את הכתובת הסופית, עם זוגות `name=value` אחרי `?`.
- `headers={'User-Agent': ...}` אומרת מי שואל, ו־`timeout` מגביל את ההמתנה. תוכנית מנומסת שואלת מעט שאלות, אחת בכל פעם, ושומרת את התשובות.
- בודקים קודם את `response.status_code`: 200 היא תשובה, ו־429 אומרת להמתין. שירות יכול גם לענות 200 עם שגיאה בתוך ה־JSON.
- `response.json()` מחזירה מילונים ורשימות, ומגיעים לערך עם מפתחות וציינים: `data['query']['pages'][0]['title']`.
- תשובה יכולה להיות חלק מהנתונים: המפתח `continue` אומר שיש עוד, וספירה של התשובה אינה ספירה של הדף.
- מפתח שחסר עוצר את התוכנית עם `KeyError`; `get(key, default)` מחזירה ברירת מחדל, וברירת המחדל צריכה להיות מהסוג שהצעד הבא צריך.

</div>

<div dir="rtl">

**השירות החי.** התא שלמטה יכול לבקש מוויקיפדיה האנגלית עצמה את התקציר של הדף על מערך שערים לוגיים שמתכנתים בשטח, עם סוכן משתמש שאומר איך להגיע אלינו, עם `timeout` ועם בדיקה של קוד המצב. הוא כבוי (`RUN_LIVE = False`): כיתה שלמה שמריצה אותו יחד שולחת הרבה בקשות מכתובת אחת, וויקיפדיה עונה לתוכנות ששולחות יותר מדי בקוד המצב 429. אדם אחד, פעם אחת, יכול לשנות ל־`RUN_LIVE = True` כדי לראות את השירות האמיתי. שאר המחברת עובד על העותקים השמורים.

</div>

In [23]:
RUN_LIVE = False  # switch to True to ask Wikipedia itself (one request)
if RUN_LIVE:
    live_headers = {'User-Agent': 'IntroDataScienceCourse/1.0 (teaching example; contact via dvirross.com)'}
    live = requests.get('https://en.wikipedia.org/w/api.php', params=summary_params, headers=live_headers, timeout=30)
    if live.status_code == 429:
        print('Too many requests (status 429). Wait', live.headers.get('Retry-After', 'a while'), 'seconds, do not try again now, and use the saved copy.')
    elif live.status_code != 200:
        print('The service answered with status', live.status_code, '. Use the saved copy.')
    else:
        live_page = live.json()['query']['pages'][0]
        print(live_page['title'], len(live_page['extract']))
else:
    print('The live cell is off. Set RUN_LIVE = True to run it.')